# Edge Weight Prediction — XGBoost → ONNX

Train an XGBoost model to predict co-occurrence edge weights from features,
then export to ONNX for Go inference.

**Input**: CSV with columns: `cooccCount,dfA,dfB,totalDocs,avgWeight,npmi,llr,dice,logCooccCount,dfRatio,target_weight`
**Output**: `model.onnx`

In [ ]:
!pip install -q xgboost onnxmltools onnx skl2onnx pandas scikit-learn

In [ ]:
import os

OUTPUT_DIR = "/kaggle/working/models/edge_weight_v1"  # or "./models/edge_weight_v1" local
DATA_PATH = "/kaggle/input/edge-features/edge_features.csv"  # uploaded as Kaggle Dataset
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import pandas as pd

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df)} rows")
df.head()

In [ ]:
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

FEATURES = ["cooccCount", "dfA", "dfB", "totalDocs", "avgWeight", "npmi"]
TARGET = "target_weight"

X = df[FEATURES].values.astype(np.float32)
y = df[TARGET].values.astype(np.float32)

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

model = xgb.XGBRegressor(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    objective="reg:squarederror",
    tree_method="hist",
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=20)

preds = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, preds))
print(f"Validation RMSE: {rmse:.4f}")

In [ ]:
import onnx
from onnxmltools import convert_xgboost
from onnxmltools.convert.common.data_types import FloatTensorType

initial_types = [("input", FloatTensorType([1, len(FEATURES)]))]
onnx_model = convert_xgboost(model, initial_types=initial_types)

onnx_path = os.path.join(OUTPUT_DIR, "model.onnx")
onnx.save(onnx_model, onnx_path)
print(f"Saved ONNX model to {onnx_path}")
print(f"Model size: {os.path.getsize(onnx_path) / 1024:.1f} KB")

In [ ]:
# Validate ONNX output matches XGBoost
import onnxruntime as ort

sess = ort.InferenceSession(onnx_path)
onnx_preds = sess.run(None, {"input": X_val[:10]})[0].flatten()
xgb_preds = model.predict(X_val[:10])

for i in range(10):
    print(f"  XGB: {xgb_preds[i]:.4f}  ONNX: {onnx_preds[i]:.4f}  diff: {abs(xgb_preds[i]-onnx_preds[i]):.6f}")

max_diff = np.max(np.abs(xgb_preds - onnx_preds))
print(f"\nMax difference: {max_diff:.6f}")
assert max_diff < 1e-4, "ONNX output differs too much from XGBoost!"